In [1]:
# Check GPU allocated (T4 / V100 / A100)
!nvidia-smi

# Install required NLP libraries cleanly without version conflicts
!pip install -q transformers accelerate tweet-preprocessor wordninja emoji

Sun Sep 27 11:44:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone repository into /kaggle/working/ZeroStance
%cd /kaggle/working
!rm -rf ZeroStance
!git clone https://github.com/chenyez/ZeroStance.git
%cd /kaggle/working/ZeroStance

/content
Cloning into 'ZeroStance'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 229 (delta 4), reused 0 (delta 0), pack-reused 218 (from 2)
Receiving objects: 100% (229/229), 61.98 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (212/212), done.
/content/ZeroStance


In [3]:
import os
from transformers import AutoTokenizer, AutoModel, AutoConfig

model_name = "vinai/bertweet-large"
save_dir = "/kaggle/working/model_hub/bertweet-large"
os.makedirs(save_dir, exist_ok=True)

print("Downloading and caching BERTweet-large model...")
tokenizer = AutoTokenizer.from_pretrained(model_name, normalization=True)
config = AutoConfig.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

tokenizer.save_pretrained(save_dir)
config.save_pretrained(save_dir)
model.save_pretrained(save_dir)

print(f"Successfully cached to: {save_dir}")

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.42GB            

...

[transformers] RobertaModel LOAD REPORT from: vinai/bertweet-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Successfully cached to: /content/model_hub/bertweet-large


In [4]:
import re

# 1. Update batch_size to 8 to avoid OOM
config_path = "/kaggle/working/ZeroStance/config/config-roberta_large.txt"
with open(config_path, "r") as f:
    config_text = f.read()
config_text = re.sub(r"batch_size:\s*\d+", "batch_size:8", config_text)
with open(config_path, "w") as f:
    f.write(config_text)

# 2. Patch AdamW in BOTH files where it occurs
for file_path in ["/kaggle/working/ZeroStance/src/utils/model_utils.py", "/kaggle/working/ZeroStance/src/train_model_v2.py"]:
    with open(file_path, "r") as f:
        c = f.read()
    c = c.replace("from transformers import AdamW", "from torch.optim import AdamW")
    with open(file_path, "w") as f:
        f.write(c)

print("✅ Batch size set to 8 and AdamW patched in all files!")

✅ Batch size set to 8 and AdamW patched in all files!


In [5]:
# Patch deprecated AdamW in train_model_v2.py
train_file = "/kaggle/working/ZeroStance/src/train_model_v2.py"
with open(train_file, "r") as f:
    code = f.read()

code = code.replace("from transformers import AdamW", "from torch.optim import AdamW")

with open(train_file, "w") as f:
    f.write(code)

print("✅ Patched train_model_v2.py successfully!")

✅ Patched train_model_v2.py successfully!


In [6]:
# Patch tokenizer.batch_encode_plus in data_helper.py
data_helper_path = "/kaggle/working/ZeroStance/src/utils/data_helper.py"
with open(data_helper_path, "r") as f:
    code = f.read()

# Replace obsolete batch_encode_plus with direct tokenizer call
code = code.replace("tokenizer.batch_encode_plus(", "tokenizer(")

with open(data_helper_path, "w") as f:
    f.write(code)

print("✅ Patched data_helper.py to modern tokenizer syntax!")

✅ Patched data_helper.py to modern tokenizer syntax!


In [7]:
# Patch np.Inf to np.inf in pytorchtools.py for NumPy 2.0
tool_path = "/kaggle/working/ZeroStance/src/pytorchtools.py"
with open(tool_path, "r", encoding="utf-8", errors="ignore") as f:
    code = f.read()

code = code.replace("np.Inf", "np.inf")

with open(tool_path, "w", encoding="utf-8") as f:
    f.write(code)

print("✅ Patched pytorchtools.py to np.inf successfully!")

✅ Patched pytorchtools.py to np.inf successfully!


In [8]:
%cd /kaggle/working/ZeroStance/src

!python train_model_v2.py \
  vast ibm30k covid19 semeval2016 wtwt pstance \
  -c ../config/config-roberta_large.txt \
  -s 0 \
  -d 0.1 \
  -train ../data/ \
  -dev ../data/ \
  -test ../data/ \
  -clipgrad True \
  -step 3 \
  --earlystopping_step 3 \
  -dataset chatgpt_carto_bertweet_var_0.99_seed0 \
  --leave_one_out 0 \
  -lr1 1e-5 \
  -lr2 1e-5


/kaggle/working/ZeroStance/src
PLM lr: 1e-05
FC lr: 1e-05
Batch size: 8
Dropout: 0.1
Early stopping patience: 3
Clip gradient: True
Datasets used in leave-one-out setting: ['vast', 'ibm30k', 'covid19', 'semeval2016', 'wtwt', 'pstance']
############################################################
Current random seed: 0
Size of train set: 21356
Size of val set: 5400
Size of test set: 15018
Loading data
BERTweet is used as the stance classifier.
Loading weights: 100% 391/391 [00:00<00:00, 4350.23it/s]
Train setup has finished!
Path: ./checkpoints/chatgpt_carto_bertweet_var_0.99_seed0/, exists.
Early stopping occurs when the loss does not decrease after 3 steps.
############################################################
Epoch: 0
At step: 890
Train loss:  0.21411795665658842
Val loss:  0.23546213864829582
Test loss:  1.4760373486675735
training classification_report for step: 890
              precision    recall  f1-score   support

     Against     0.9070    0.9563    0.9310      7188
 

In [9]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import sys
sys.path.append("/kaggle/working/ZeroStance/src")

from utils import modeling, model_utils
import utils.preprocessing as pp
from utils.data_helper import convert_data_to_ids, data_loader
from transformers import AutoTokenizer
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load slang & dictionary mappings
with open("/kaggle/working/ZeroStance/src/noslang_data.json", "r") as f:
    data1 = json.load(f)
data2 = {}
with open("/kaggle/working/ZeroStance/src/emnlp_dict.txt", "r") as f:
    for line in f:
        row = line.split('\t')
        data2[row[0]] = row[1].rstrip()
norm_dict = {**data1, **data2}

# Load tokenizer and best model weights
model_path = "/kaggle/working/ZeroStance/src/checkpoints/chatgpt_carto_bertweet_var_0.99_seed0/RoBERTa_seed0.pt"
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/model_hub/bertweet-large", local_files_only=True)
model = modeling.roberta_large_classifier(num_labels=3, model_select="RoBERTa", dropout=0.1).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# Benchmark datasets
test_datasets = ["vast", "ibm30k", "covid19", "semeval2016", "wtwt", "pstance"]
results = {}
config = {'max_tok_len': '250', 'batch_size': '16'}

print("Evaluating ZeroStance across all 6 benchmark datasets:\n" + "="*55)
for d_name in test_datasets:
    test_file = f"/kaggle/working/ZeroStance/data/{d_name}/raw_test_all_onecol.csv"
    if not os.path.exists(test_file):
        print(f"Skipping {d_name} (file not found)")
        continue

    x_test, y_test, x_target = pp.clean_all(test_file, norm_dict, d_name, d_name)
    encoded = convert_data_to_ids(tokenizer, x_target, x_test, y_test, config)
    loader, _ = data_loader(encoded, 16, "RoBERTa", "test")

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            inputs = {
                'input_ids': batch[0].to(device),
                'attention_mask': batch[1].to(device)
            }
            outputs = model(**inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch[-1].numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro') * 100
    results[d_name] = round(macro_f1, 2)
    print(f"Dataset: {d_name:<12} | Macro-F1: {macro_f1:.2f}%")

avg_f1 = np.mean(list(results.values()))
print("="*55)
print(f"Average Open-Domain Macro-F1: {avg_f1:.2f}%\n")

Evaluating ZeroStance across all 6 benchmark datasets:
Dataset: vast         | Macro-F1: 64.18%
Dataset: ibm30k       | Macro-F1: 61.74%
Dataset: covid19      | Macro-F1: 63.27%
Dataset: semeval2016  | Macro-F1: 65.83%
Dataset: wtwt         | Macro-F1: 60.92%
Dataset: pstance      | Macro-F1: 62.46%
Average Open-Domain Macro-F1: 63.07%


In [10]:
label_map = {0: "AGAINST ❌", 1: "FAVOR ✅", 2: "NEUTRAL / NONE ⚪"}

def predict_stance(text, target):
    # Preprocess text and target
    clean_text = pp.data_clean(text, norm_dict)
    clean_target = pp.data_clean('I am in favor of ' + target + " ! ", norm_dict)

    # Tokenize input
    encoded = tokenizer(
        [[' '.join(clean_text), ' '.join(clean_target)]],
        add_special_tokens=True,
        max_length=250,
        padding='max_length',
        return_attention_mask=True,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        'input_ids': encoded['input_ids'].to(device),
        'attention_mask': encoded['attention_mask'].to(device)
    }

    with torch.no_grad():
        logits = model(**inputs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))

    print(f"\n--- Stance Prediction ---")
    print(f"Target:      {target}")
    print(f"Text:        {text}")
    print(f"Stance:      {label_map[pred_idx]}")
    print(f"Confidence:  Against: {probs[0]:.2%}, Favor: {probs[1]:.2%}, Neutral: {probs[2]:.2%}\n")


# === Try out custom examples ===
predict_stance(
    text="Vaccines have been proven safe and effective by countless clinical trials and health organizations.",
    target="COVID-19 vaccination"
)

predict_stance(
    text="Nuclear power generation creates toxic waste that will remain dangerous for thousands of years.",
    target="Nuclear power expansion"
)

predict_stance(
    text="The concert started around 8 PM and tickets were completely sold out.",
    target="Abolishing capital punishment"
)

--- Stance Prediction ---
Target:      COVID-19 vaccination
Text:        Vaccines have been proven safe and effective by countless clinical trials and health organizations.
Stance:      FAVOR ✅
Confidence:  Against: 2.31%, Favor: 96.84%, Neutral: 0.85%

--- Stance Prediction ---
Target:      Nuclear power expansion
Text:        Nuclear power generation creates toxic waste that will remain dangerous for thousands of years.
Stance:      AGAINST ❌
Confidence:  Against: 95.72%, Favor: 2.91%, Neutral: 1.37%

--- Stance Prediction ---
Target:      Abolishing capital punishment
Text:        The concert started around 8 PM and tickets were completely sold out.
Stance:      NEUTRAL / NONE ⚪
Confidence:  Against: 7.44%, Favor: 5.82%, Neutral: 86.74%


In [11]:
import os
import shutil
from pathlib import Path

backup = Path("/kaggle/working/ZeroStance_Backup")
if backup.exists():
    shutil.rmtree(backup)
backup.mkdir()

shutil.copytree(
    "/kaggle/working/ZeroStance",
    backup / "ZeroStance",
    ignore=shutil.ignore_patterns("model_hub", "__pycache__", "*.pyc")
)

zip_path = shutil.make_archive(
    "/kaggle/working/ZeroStance_Backup", "zip", "/kaggle/working/ZeroStance_Backup"
)

size_mb = os.path.getsize(zip_path) / (1024**2)
print("Backup created successfully ✅")
print(f"ZIP size: {size_mb:.2f} MB")
print("BERTweet-large model excluded.")
print("ZeroStance source + checkpoints/results included.")


Backup created successfully ✅
ZIP size: 428.63 MB
BERTweet-large model excluded.
ZeroStance source + checkpoints/results included.
